# Wave 5 + 6i — four gross-code LPU operations, fail-fast

Short report on the Wave-5 campaigns run on BB(12) = [[144,12,12]] (Tour-de-Gross LPU), plus the
Wave-6i inter-module gate. The first three are complete on the local 24-core box; the fourth is
built and validated but not yet swept:

| run | outdir | status |
|---|---|---|
| **LPU idle** — bare LPU held for `rounds` QEC cycles | `runs/framework/bb144/lpu_idle` | complete 2026-07-24 12:57 |
| **Shift automorphism** — the 14-timestep swap circuit, `C` repeats, `delta = y` | `runs/framework/bb144/automorphism` | complete 2026-07-24 21:08 |
| **Joint Pauli** — `Ybar_1` measured through the whole LPU | `runs/framework/bb144/joint_pauli` | complete 2026-07-25 01:10 |
| **Inter-module** — `Xbar_1 (x) Xbar_1` across two modules via the code-code adapter | `runs/framework/bb144/inter_module` | **not run** — validated only, see §5 |

Common setup for the three completed runs: `p_ref = 5e-3`, seed 42, `adaptive_shots_max = 3000`,
full-DEM (`-XYZ`, non-CSS) **Relay-BP with the tuned-cheap `num_sets=20`** — *not* the paper's 600.
These are fail-fast numbers: an upper bound on error, a lower bound on what the operation can do.

**Read the inter-module entries with care.** Everywhere it appears in §1–§3 the data is
*validation* at C = 3 / `d_init` = 2, not a production sweep of the C = 10 circuit, and it is
marked as such at every appearance. What it establishes is qualitative — no low-weight floor, and
the retraction of the `obs0` "floor" reported against it — not values to quote alongside the three
production rows. §5 has the full account.

In [ ]:
# src/ is an editable install (`pip install -e .`) - modules import directly, no sys.path needed.
import json
import pathlib
import numpy as np
import matplotlib.pyplot as plt

from lambda_analysis import (load_run, fill_spectrum, reweight_filled, rw_stats, eps_stats,
                             mass_window_p_max, zero_bin_fraction)

RUNS = pathlib.Path("../../runs/framework/bb144")
OPS = {"lpu_idle": "LPU idle", "automorphism": "shift automorphism", "joint_pauli": "Y1 joint Pauli"}
colors = {"lpu_idle": "#2c7fb8", "automorphism": "#d95f0e", "joint_pauli": "#756bb1",
          "inter_module": "#2ca25f"}

runs = {k: load_run(RUNS / k) for k in OPS}
filled = {k: fill_spectrum(r.spectrum) for k, r in runs.items()}
P_TARGET = 1e-3          # the campaign's reporting point
print({k: (len(r.spectrum.weights), r.cycles) for k, r in runs.items()})

# Wave 6i inter-module validation (section 5). Lives in experiments/ rather than runs/,
# which is gitignored, so this notebook renders with no production run present.
W6VAL = json.loads(
    pathlib.Path("../../experiments/tour_de_gross/data/wave6i_intermodule_validation.json")
    .read_text(encoding="utf-8"))
IM_CYCLES = 10           # cycles_of routes inter_module -> lpu_C, and both configs set lpu_C: 10

## 1. Summary

`eps` is per-cycle under `cycles_of`: QEC rounds for idle (12), the repeated-measurement rounds
`lpu_C` for the automorphism and `Ybar_1` (10 each). `headroom` is the rule-of-three exposure from
sampled-but-empty bins — reweighted values are lower bounds, headroom is how far they could rise.

The two **inter-module** rows are a different kind of entry and are marked `*`. Their `N_exp` is
real — it comes analytically from the production DEM, and it is what fixed the frozen weight
blocks — but the circuit has **never been swept**, so every *sampled* column is `-`. Its
validation LER is deliberately **not** put in this table: that measurement is at C = 3 / `d_init`
= 2, and dropping a different geometry into a column of production numbers is precisely the
apples-to-oranges error that produced the retracted `obs0` "floor". Section 5 has it, in context.

In [ ]:
hdr = f"{'op':<20}{'N_exp':>10}{'cyc':>5}{'D':>5}{'w0':>5}{'LER(1e-3)':>12}{'+-se':>10}{'head':>10}{'eps':>11}{'zero%':>7}"
print(hdr); print("-" * len(hdr))
for k, label in OPS.items():
    r, s = runs[k], filled[k]
    L, se, head = rw_stats(s, P_TARGET)
    eps = eps_stats(s, P_TARGET, r.cycles)[0]
    D = r.distance["distance"] if r.distance else None
    w0 = r.distance["onset"] if r.distance else None
    print(f"{label:<20}{r.spectrum.n_expanded:>10d}{r.cycles:>5d}"
          f"{(D if D else '-'):>5}{(w0 if w0 else '-'):>5}"
          f"{L:>12.2e}{se:>10.1e}{head:>10.1e}{eps:>11.2e}{100*zero_bin_fraction(s):>7.0f}")

# Wave 6i inter-module. The circuit is SIZED (N_exp is analytic, straight from the production
# DEM) but never SWEPT, so every sampled quantity is '-'. Technique II is out for the same
# reason as the automorphism/Ybar_1 rows. Deliberately NOT filled with the C=3 validation
# LER: that is a different geometry and mixing it into this table is exactly the
# apples-to-oranges error that produced the retracted obs0 "floor". See section 5.
for leg in ("r1", "r10"):
    sz = W6VAL["production_sizing"][leg]
    print(f"{'inter-module ' + leg + ' *':<20}{sz['n_expanded']:>10d}{IM_CYCLES:>5d}"
          f"{'-':>5}{'-':>5}{'-':>12}{'-':>10}{'-':>10}{'-':>11}{'-':>7}")
print("  * sized, not swept: weights_range frozen to "
      f"[1,{W6VAL['production_sizing']['r1']['w_hi']}] / "
      f"[1,{W6VAL['production_sizing']['r10']['w_hi']}]; no production spectrum exists yet.")

print("\nTechnique-I ansatz (f5, all unpinned):")
for k, label in OPS.items():
    a = runs[k].ansatz["params"]
    print(f"  {label:<20} w0={a['w0']:>5.1f}  f0={a['f0']:.2e}  g1={a['gamma1']:>5.2f}  "
          f"g2={a['gamma2']:>5.2f}  wc={a['wc']:>6.1f}   (cost {runs[k].ansatz['cost']:.1f}, "
          f"{runs[k].ansatz['n_points']} pts)")
print(f"  {'inter-module':<20} no fit - Technique I needs the sweep (Technique II is out of "
      f"reach at this column count, as for the automorphism and Ybar_1)")

## 2. The importance-sampled failure spectrum

`f(w)` = fraction of weight-`w` fault configurations the decoder gets wrong — the raw measurement
everything else is built on. The reweighted LER of the next section is just `sum_w f(w) P(w|p)`,
so this is where a sick operation shows itself first.

Points are the sampled bins with binomial error bars; open markers on the floor are
zero-failure bins (plotted at the rule-of-three upper limit `3/T`, i.e. an upper bound, not a
measurement). Lines are the Technique-I `f5` fits. Dashed verticals mark the mean fault weight at
p = 1e-3, `N_exp * q_base * (p/p_ref)` — the weights that actually carry the mass there.

**Green diamonds are the inter-module gate, and they are not the same kind of data.** They come
from the C = 3 / `d_init` = 2 validation probe at `w = 1..6`, not from a production sweep of the
C = 10 circuit, so they are drawn hollow and given no fit line. Read them qualitatively — *no
low-weight floor, matching the `Ybar_1` baseline* — and do not read values off them against the
three production spectra. The shaded band is the frozen production weight block
(`weights_range` = [1, 1674] for `r1`) and the dash-dotted vertical is where its mass will sit at
p = 5e-3 (`mu` ≈ 1518); that is the region its sweep will actually sample once it runs.

In [ ]:
from importance_sampling import failure_spectrum_ansatz

fig, ax = plt.subplots(figsize=(6.8, 4.6))
a_sat = 1 - 2.0 ** -12                      # K = 12 saturation, f(w) -> 1 - 2^-K

for k, label in OPS.items():
    r = runs[k]
    w = np.asarray(r.spectrum.weights, float)
    T = np.asarray(r.spectrum.trials, float)
    F = np.asarray(r.spectrum.failures, float)
    f = F / T
    se = np.sqrt(np.clip(f * (1 - f), 0, None) / T)
    hit, zero = F > 0, F == 0
    ax.errorbar(w[hit], f[hit], yerr=se[hit], fmt="o", ms=3.5, lw=.8, color=colors[k],
                label=f"{label}  ({int(hit.sum())}/{len(w)} bins with failures)")
    ax.plot(w[zero], 3.0 / T[zero], "v", ms=3.5, mfc="none", mew=.8, color=colors[k], alpha=.55)

    p = runs[k].ansatz["params"]
    wg = np.linspace(max(p["w0"], 1), w.max(), 400)
    ax.plot(wg, failure_spectrum_ansatz(wg, p["w0"], p["f0"], a_sat, model="f5",
                                        gamma1=p["gamma1"], gamma2=p["gamma2"], wc=p["wc"]),
            "-", lw=1.2, color=colors[k], alpha=.75)
    w_bar = r.spectrum.n_expanded * r.spectrum.q_base * (P_TARGET / r.p_ref)
    ax.axvline(w_bar, color=colors[k], ls="--", lw=.9, alpha=.5)

# --- Wave 6i inter-module: VALIDATION ONLY, and a DIFFERENT GEOMETRY -------------------
# C=3/d_init=2 vs the C=10-class production runs above, so these points are NOT on the same
# footing as the three spectra: they probe w=1..6 of a shallower circuit. Drawn hollow with
# no fit line to keep that distinction visible. What they establish is qualitative and
# geometry-independent - no low-weight floor - not a spectrum to read values off.
_fs = W6VAL["failure_spectrum"]
_W = np.asarray(_fs["weights"], float)
_T = _fs["conditions"]["T_per_weight"]
_F = np.asarray(_fs["series"]["inter_closure"]["failures"], float)
_hit, _zero = _F > 0, _F == 0
_f = _F / _T
ax.errorbar(_W[_hit], _f[_hit], yerr=np.sqrt(_f[_hit] * (1 - _f[_hit]) / _T),
            fmt="D", ms=5, lw=0, elinewidth=.9, mfc="none", mew=1.3, color=colors["inter_module"],
            label="inter-module (validation, C=3 - not a production spectrum)")
ax.plot(_W[_zero], np.full(int(_zero.sum()), 3.0 / _T), "D", ms=5, mfc="none", mew=.9,
        color=colors["inter_module"], alpha=.45)
# frozen production weight block - where its sweep WILL sample once it runs
ax.axvspan(1, W6VAL["production_sizing"]["r1"]["w_hi"], color=colors["inter_module"],
           alpha=.05, zorder=0)
ax.axvline(W6VAL["production_sizing"]["r1"]["mu_p_hi"], color=colors["inter_module"],
           ls="-.", lw=.9, alpha=.6)

ax.axhline(a_sat, color="k", ls=":", lw=.8)
ax.text(900, a_sat * 1.1, r"$1-2^{-K}$", fontsize=7, va="bottom", ha="right")
ax.set(xscale="log", yscale="log", xlabel="fault weight $w$", ylabel="$f(w)$",
       ylim=(1e-4, 4), xlim=(1, 1800),
       title="Measured failure spectra (points) and $f5$ fits (lines)")
ax.legend(fontsize=6.5, loc="upper left"); ax.grid(alpha=.25, which="both")
fig.tight_layout()

## 3. Logical error rate

Points = importance-sampled spectrum, gap-filled, reweighted; lines = the Technique-I `f5` fit.
Curves are cut at `mass_window_p_max` (4 sigma of binomial mass inside the sampled window) —
beyond it reweighting is no longer unbiased.

**The green diamonds and purple crosses are direct Monte Carlo, not importance sampling**, at
C = 3 / `d_init` = 2 — a different estimator on a different circuit depth. They are on this axis
for one reason: they are the measurement that produced, and then retired, the reported `obs0`
"floor". The inter-module gate reads ~0.4 at p = 5e-3 and **0.020 ± 0.010 at p = 1e-3** — a
threshold crossing, not a floor. The `Ybar_1` crosses are the control that settles it: the
*known-good* operation, run at the same settings, floors just as hard at p = 5e-3 (0.35). At
~106 expected faults per shot against `d ~ 10`, LER ~ 0.5 there is arithmetic, not a defect.
Do not compare either against the IS curves.

In [ ]:
fig, ax = plt.subplots(figsize=(6.4, 4.4))

for k, label in OPS.items():
    r, s = runs[k], filled[k]
    res = np.load(RUNS / k / "result.npz")
    pg = res["p_values"]
    ok = pg <= mass_window_p_max(s)
    ax.plot(pg[ok], reweight_filled(s, pg[ok]), "o", ms=4, color=colors[k], label=f"{label} (IS)")
    ax.plot(res["ansatz_p"], res["ansatz_P"], "-", lw=1.4, color=colors[k], alpha=.75,
            label=f"{label} (ansatz)")

# --- Wave 6i inter-module: DIRECT MONTE CARLO at C=3/d_init=2 --------------------------
# NOT importance-sampled and NOT the production geometry, so these are not comparable to
# the curves above - they are here to show the shape that retired the reported "floor":
# ~0.4 at p=5e-3 (far above threshold, ~106 expected faults/shot) but 0.02 at p=1e-3.
# A single high-p MC point cannot distinguish a broken observable from an over-noised
# circuit; that is what section 5's f(w) is for.
_mc = [q for q in W6VAL["monte_carlo_ler"]["points"] if q["circuit"] == "inter_module"]
_p = np.array([q["p"] for q in _mc], float)
_L = np.array([q["ler"] for q in _mc], float)
_se = np.array([q["se"] for q in _mc], float)
ax.errorbar(_p, _L, yerr=_se, fmt="D", ms=5.5, lw=0, elinewidth=1.1, capsize=3,
            mfc="none", mew=1.4, color=colors["inter_module"],
            label="inter-module (direct MC, C=3 - not IS, not production)")

_y1 = [q for q in W6VAL["monte_carlo_ler"]["points"]
       if q["circuit"] == "y1_baseline" and not q["idle"]]
ax.errorbar([q["p"] for q in _y1], [q["ler"] for q in _y1],
            yerr=[q["se"] for q in _y1], fmt="x", ms=7, lw=0, elinewidth=1.1, capsize=3,
            color=colors["joint_pauli"], alpha=.9,
            label="Ybar_1 same-setup MC control (floors just as hard)")

ax.axvline(P_TARGET, color="k", ls=":", lw=.8)
ax.set(xscale="log", yscale="log", xlabel="physical error rate p", ylabel="logical error rate",
       ylim=(1e-12, 2), title="Wave-5 gross-code LPU operations (relay num_sets=20)")
ax.legend(fontsize=6.5, ncol=1, loc="lower right"); ax.grid(alpha=.25, which="both")
fig.tight_layout()

## 4. What the runs say

**Cost tracks DEM size.** Expanded mechanism counts are 5.2e5 (idle) / 1.7e6 (automorphism) /
2.9e6 (`Ybar_1`), and the LER at p = 1e-3 orders the same way: ~2e-5, ~6e-3, ~0.5. `Ybar_1` is the
whole LPU in one operation, so at fixed p it carries roughly 6x the idle fault load; its shallow
`gamma1 = 4.55` is that, not a decoder pathology. The inter-module gate extends the trend and sits
at the top of it — **4.5e6 / 4.7e6** expanded mechanisms, 1.55x / 1.62x `Ybar_1` — which is what two
modules plus the adapter costs. Its LER is not yet measurable at that geometry, but the ordering
is already fixed by the DEM.

**Idle is the only run with Technique II**: D = 10 (bound), `w0` = 5, `f0*` = 4.8e-15, which meets
the paper's `d <= 10` bound for the LPU idle. Technique II was **dropped** from the automorphism
and `Ybar_1` configs — BP-OSD runs ~2 h/decode at 2e5+ columns (the smoke stalled at 6/50 trials
in 10 h). Their `result.npz` therefore carries `distance = 0`, `onset = nan` **by design**, and
their fits are `w0`/`f0`-free. The paper likewise has no `Ybar_1` row in Table 2. The inter-module
config drops it for the same reason, with an extra one: `compute_distance` returns a spurious 1 on
these deformed non-CSS circuits.

**Both IS sweeps ended on the stop rule, not exhaustion** (3 consecutive zero-failure bins at
`shots_max = 3000`): the automorphism skipped 14 lower weights, `Ybar_1` skipped 1. Skipped
weights sit below the onset and contribute ~0.

### Three caveats before quoting any of this

1. **Never read `is_P_logical` straight out of `result.npz`.** The raw IS sum runs only over
   *sampled* weights, so a strided tail saturates at exactly `1/stride` instead of going to 1:
   idle is stride 1 (-> 1.0), the automorphism stride 4 (-> 0.250), `Ybar_1` stride 6 (-> 0.1667).
   It is low by ~stride at *every* p. `f(w)` itself is healthy (1.0 in the top bins). The cell
   below shows gap-filling closing the gap against the ansatz.
2. **Decoder handicap.** `num_sets = 20`, not the paper's 600. Combined with `p_ref = 5e-3` vs the
   paper's 1e-4 importance-sampling prior, these numbers are directionally comparable to Table 3,
   not numerically.
3. **The inter-module entries are not production numbers.** Only `N_exp` and the frozen weight
   blocks are properties of the C = 10 circuit; everything else shown for it is C = 3 validation.
   Nothing in this section's cost or ordering discussion should be quoted for it beyond the DEM
   size.

In [ ]:
# The stride artifact, and gap-filling as the fix.
print(f"{'op':<20}{'stride':>8}{'raw sat.':>10}{'1/stride':>10}   | at p=1e-3: {'raw':>10}{'filled':>10}{'ansatz':>10}")
for k, label in OPS.items():
    r, s = runs[k], filled[k]
    res = np.load(RUNS / k / "result.npz")
    w = np.asarray(r.spectrum.weights)
    stride = int(np.median(np.diff(w)))
    i = int(np.argmin(np.abs(res["p_values"] - P_TARGET)))
    print(f"{label:<20}{stride:>8d}{res['is_P_logical'][-1]:>10.4f}{1/stride:>10.4f}   |            "
          f"{res['is_P_logical'][i]:>10.2e}{reweight_filled(s, [P_TARGET])[0]:>10.2e}"
          f"{res['ansatz_P'][i]:>10.2e}")

## 5. Inter-module gate — VALIDATED (production run still pending)

The fourth operation, and the one Wave 6i is built around: a **gross-to-gross adapter** joining two
[[144,12,12]] modules (A frame 0, B frame 378) to measure `Xbar_1 (x) Xbar_1` across them. Merged to
`main`; branch was `wave6-intermodule`.

**Built and green:** `AdapterGraph` + cross-module `U_B` derivation, `build_adapter_cycle` (11 bridge
Bell checks + 10 cross-module `U_B` checks), `build_joint_x1x1_circuit`; wired into
`experiment_runner` and `lambda_analysis`; configs `gross_intermodule_{r1,r10}.yaml`. Gates **E1**
(p=0 determinism), **E2** (obs0 reproduces the MPP reference), **E4** (DEM + decoder) all pass.
Both paper-ambiguous knobs were resolved empirically: the bridge identification is **CX** (CZ
anticommutes with the X-vertex checks), and the cross-module `U_B` check takes **both** physical
copies of each bridge qubit.

### The reported `obs0` floor was a MISDIAGNOSIS — retired 2026-07-27

The earlier draft of this section recorded an open blocker: *"`obs0` floors at LER ~ 0.4, a bridge
Bell gauge issue."* That was a real measurement read the wrong way.

> **A Monte-Carlo LER at `p_ref` cannot distinguish a broken observable from a circuit operating far
> above threshold.** It is not a diagnostic for observable health.

Three independent lines of evidence retire it:

1. **The circuit is far above threshold at that point.** Straight from the DEM it sees **105.7
   expected faults per shot** at p = 5e-3 against `d ~ 10`. LER ~ 0.5 there is arithmetically
   unavoidable for *any* circuit of this size.
2. **The known-good baseline floors just as hard.** `Ybar_1` — validated, in this very report —
   gives LER 0.35 (idle off) and 0.51 (idle on) at the same point.
3. **It decodes fine below threshold.** The inter-module circuit reaches **LER 0.020 ± 0.010 at
   p = 1e-3**. That is a threshold crossing, not a floor.

The correct, **p-independent** health check is the low-weight failure spectrum `f(w)` — the same
quantity section 2 plots, and exactly what `techniques: [IS, I]` measures. It is now a standing gate
in the runbook, alongside the weight-1 degeneracy scan promoted after the `Ybar_1` framing bug: **the
degeneracy scan catches *undetectable* faults, `f(w)` catches *miscorrected* ones.**

Two cautions carried forward, both learned the hard way here:

* **Compare against the validated baseline, never against zero.** `Ybar_1` is itself nonzero at
  w = 3, 4, 6 (1, 1, 3 per 400) — decoder miscorrection at the cheap 20-set relay, not undetectable
  errors. "Matches the baseline" is the standard.
* **`shortest_graphlike_error` is a misleading proxy** on these deformed non-CSS circuits: it skips
  precisely the hyperedge/gauge errors at issue, and `compute_distance` returns a spurious 1 (the
  validated `Ybar_1` circuit does too). Where MC, graphlike distance and `f(w)` disagree, **trust
  `f(w)`.**

### `close_cycles` — correct, but not the cure

A fix landed alongside: terminal Z-cycle closure detectors for both modules' `U_l` cycles and the 10
cross-module `U_B` checks, each XORing the last merged round's cycle record against its edge
readouts — the `prod m_e = +1` boundary `build_joint_pauli_circuit` already used. Without them the
final merged round's Z-cycle information was simply discarded. It adds 20 detectors (2722 -> 2742
here; 10883 -> 10903 at production geometry), all verified deterministic at p = 0.

**Honest scope: it is a correctness improvement, not what removed the floor** — the pre-closure
circuit is equally clean at low weight, as the cells below show. The X-type bridge Bell checks have
no analogue (a Z-basis edge readout cannot cancel them) and still terminate into `obs0`.

### What is still pending

The numbers below are **validation** at reduced geometry (C = 3, `d_init` = 2), not the production
campaign (C = 10, `d_init` = 12), which has not been run. `weights_range` is now frozen for it from
an idle-ON sizing probe — **r1 [1, 1674], r10 [1, 1742]** — replacing a `[1, 900]` placeholder that
would have stopped short of the dominant mass at w ≈ 1518–1583 and corrupted the high-p end.
`lpu_include_memory_obs` remains `false` pending the K=23 merged-graph recipe, so these runs will
carry the operator observables only. See `docs/cluster_runbook.md`, section **Wave 6i**.

The second cell picks the production run up automatically once it exists.

In [ ]:
import json

# Validation measurements live in experiments/ (not runs/, which is gitignored) so this
# section renders without a production run. Reproduce with:
#   python experiments/tour_de_gross/failure_spectrum_probe.py --op inter_module ...
W6 = pathlib.Path("../../experiments/tour_de_gross/data/wave6i_intermodule_validation.json")
val = json.loads(W6.read_text(encoding="utf-8"))
fs = val["failure_spectrum"]
W = np.asarray(fs["weights"], int)
T = fs["conditions"]["T_per_weight"]

print(f"f(w) validation - C={fs['conditions']['C']} d_init={fs['conditions']['d_init']} "
      f"p={fs['conditions']['p']:g}, {fs['conditions']['decoder']}, T={T}/weight\n")
print(f"{'series':<36}" + "".join(f"w={w:<6d}" for w in W) + "  total")
for key, s in fs["series"].items():
    F = np.asarray(s["failures"], int)
    print(f"{s['label']:<36}" + "".join(f"{f:<8d}" for f in F) + f"  {F.sum()}/{len(W) * T}")

ef = val["expected_faults_per_shot"]
print("\nE[faults/shot] (DEM, decoder-independent) - why MC at p_ref cannot diagnose obs0:")
for p, a, b in zip(ef["p"], ef["inter_module"], ef["y1_baseline"]):
    print(f"  p={p:<8g} inter-module {a:>7.2f}   Y1 baseline {b:>7.2f}")

im = RUNS / "inter_module"
if (im / "spectrum.json").exists():
    r = load_run(im); s = fill_spectrum(r.spectrum)
    L, se, head = rw_stats(s, P_TARGET)
    print(f"\nPRODUCTION run present - supersedes the validation above:"
          f"\n  N_exp={r.spectrum.n_expanded}  cycles={r.cycles}  done={r.done_fraction:.0%}  "
          f"LER(1e-3)={L:.2e} +-{se:.1e} (head {head:.1e})")
else:
    print(f"\nProduction run not present at {im} - validation only.")
    print("Next: container/run_intermodule.sh (NOT written yet), then r1 + r10. Runbook 'Wave 6i'.")

In [ ]:
from scipy.stats import beta


def clopper_pearson(k, n, alpha=0.05):
    """Exact binomial CI. Counts here are 0-3 out of 400, where the normal approximation
    used in section 2 is invalid - it collapses to zero width at k=0, hiding all of the
    uncertainty in exactly the bins the conclusion rests on."""
    k = np.asarray(k, float)
    lo = np.where(k > 0, beta.ppf(alpha / 2, np.maximum(k, 1), n - k + 1), 0.0)
    hi = np.where(k < n, beta.ppf(1 - alpha / 2, k + 1, np.maximum(n - k, 1)), 1.0)
    return lo, hi


STYLE = {"y1_baseline": ("#756bb1", "o"),     # same purple as joint_pauli in section 2
         "inter_prefix": ("#999999", "s"),
         "inter_closure": ("#2ca25f", "D")}
SCALE = 1e-3

fig, ax = plt.subplots(figsize=(7.0, 4.2))
for i, (key, s) in enumerate(fs["series"].items()):
    F = np.asarray(s["failures"], float)
    f = F / T
    lo, hi = clopper_pearson(F, T)
    c, m = STYLE[key]
    off = (i - 1) * 0.16                      # nudge: the series share identical values
    ax.errorbar(W + off, f / SCALE, yerr=[(f - lo) / SCALE, (hi - f) / SCALE],
                fmt=m, ms=6, lw=0, elinewidth=1.3, capsize=3.5, color=c,
                label=f"{s['label']}  ({int(F.sum())}/{len(W) * T})")

ax.set(xlabel="fault weight $w$", ylabel=r"$f(w)$  [$\times 10^{-3}$]", xticks=W,
       ylim=(0, 22), xlim=(0.5, 6.5),
       title="Wave 6i: inter-module $f(w)$ vs the validated $\\bar{Y}_1$ baseline\n"
             "points = measured, bars = exact 95% binomial CI (T=400/weight)")
ax.legend(fontsize=7.5, loc="upper left", framealpha=.95)
ax.grid(alpha=.25, axis="y")
fig.tight_layout()

print("Every interval overlaps every other at every weight => indistinguishable.")
print(f"A k=0 bin is NOT 'zero': its 95% CI is [0, {3.0 / T:.4f}] - the rule-of-three bound,")
print(f"which is why a clean 0/{T} BOUNDS the floor at ~{1/T:.1e} rather than disproving one.")
print(f"Ruling out the ~5e-4-class floor the campaign configs reference needs T >~ 10000.")

## 6. Next

* **⚠️ rodan has NO SCHEDULER.** Discovered 2026-07-27: no `sbatch`/`srun`, and it is a *shared*
  96-core box. `experiments/slurm/submit_lpu.sh` and `submit_lpu_boost.sh` are unusable there —
  launch via detached podman with `container/run_lpu_boost.sh` instead (env-var thread capping, not
  `podman --cpus`; mounts `experiments/` as well as `runs/`, which `container/run_local.sh` does
  not, so that one would silently run the image's baked configs). Budget: 3 boost jobs × 8 threads
  = 24 of 96 cores; adding the two Wave-6i jobs makes 40 of 96.
* **Wave 5b boost pass** re-samples all three at deeper caps to push the zero-bin floor from the
  rule-of-three ~1e-3 toward ~1e-5. That is what buys `Ybar_1` a meaningful low-p tail; the paper's
  `num_sets = 600` relay and a real Technique II remain out of reach at this column count.
* **Wave 6i production run**: needs `container/run_intermodule.sh` written first (a two-line
  addition to the boost launcher), then `r1` + `r10` at the frozen weight blocks.
* **Large-T `f(w)` confirmation** for the inter-module gate — the section-5 result bounds the floor
  at ~2.5e-3, which sits *above* the ~5e-4-class floor the campaign configs reference. `T = 10000`
  closes that gap; `failure_spectrum_probe.py` is chunked, checkpointed per weight, and tops up on
  re-run rather than restarting.
* **Decoder variant**: a `decoder_p` knob (`CalibratedRelayBP`, ported from the K=4 work) as a
  separate config, to separate decoder miscalibration from circuit cost in the `Ybar_1` number.
* **Contiguous low-weight tails** for the automorphism and `Ybar_1` if their low-p ends are ever
  quoted as point values rather than through the fit.